# 2-Bosqich: ERA5 reanaliz + Sentinel-5P validatsiya

Bu notebook:
1. **ERA5 reanaliz** ma'lumotlarini yuklab olish (shamol, namlik, harorat — gridlangan)
2. **Sentinel-5P UVAI** (Aerosol Index) bilan 1-bosqichdagi label'larni validatsiya qilish
3. **Feature matrix** yaratish — ML model uchun tayyor kirish ma'lumoti

### Kerakli accountlar (bepul):
- **CDS API** (ERA5): https://cds.climate.copernicus.eu — ro'yxatdan o'ting, API key oling
- **Google Earth Engine** (Sentinel-5P): https://earthengine.google.com — signup

---

## 1. Kutubxonalar va sozlamalar

In [ ]:
!pip install cdsapi xarray netcdf4 cfgrib earthengine-api geemap pandas matplotlib seaborn --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
from datetime import datetime, timedelta

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("Kutubxonalar yuklandi!")

In [ ]:
# Yo'llar
RESULTS_DIR = "../analysis/results/"
ERA5_DIR = "../data/era5/"
SATELLITE_DIR = "../data/satellite/"

os.makedirs(ERA5_DIR, exist_ok=True)
os.makedirs(SATELLITE_DIR, exist_ok=True)

# 1-bosqich natijalarini yuklash
coords = pd.read_csv("../data/stations_coordinates.csv")

# Agar 1-bosqich CSV mavjud bo'lsa
if os.path.exists(f"{RESULTS_DIR}all_stations_labeled.csv"):
    df_labeled = pd.read_csv(f"{RESULTS_DIR}all_stations_labeled.csv", parse_dates=['date'])
    print(f"✅ 1-bosqich natijalari yuklandi: {len(df_labeled):,} qator")
else:
    print("⚠️ Avval step1 notebook'ni run qiling!")

# O'zbekiston chegaralari (ERA5 uchun)
UZB_BOUNDS = {
    'north': 46.0,
    'south': 37.0,
    'west': 56.0,
    'east': 74.0
}

## 2. ERA5 reanaliz ma'lumotlarini yuklab olish

### ERA5 nima?
ECMWF (Yevropa O'rta muddatli ob-havo prognozi markazi) tomonidan taqdim etiladigan global atmosfera reanalizi. 
- **Chastota:** soatlik
- **Grid:** 0.25° x 0.25° (~28 km)
- **Davr:** 1940-hozir
- **Bepul**

### CDS API sozlash:
1. https://cds.climate.copernicus.eu ga ro'yxatdan o'ting
2. Profile > API key ni nusxalang
3. `~/.cdsapirc` faylini yarating (quyidagi cell yordam beradi)

In [ ]:
# CDS API sozlash
# Agar birinchi marta bo'lsa — API key'ingizni kiriting:

CDS_URL = "https://cds.climate.copernicus.eu/api"
CDS_KEY = "SIZNING_API_KEY"  # <-- BU YERNI O'ZGARTIRING!

# ~/.cdsapirc faylini yaratish
cdsapirc_path = os.path.expanduser("~/.cdsapirc")

if not os.path.exists(cdsapirc_path) and CDS_KEY != "SIZNING_API_KEY":
    with open(cdsapirc_path, 'w') as f:
        f.write(f"url: {CDS_URL}\nkey: {CDS_KEY}\n")
    print(f"✅ CDS API config saqlandi: {cdsapirc_path}")
elif os.path.exists(cdsapirc_path):
    print(f"✅ CDS API config allaqachon mavjud")
else:
    print("⚠️ CDS_KEY ni o'zgartiring! (Yuqoridagi cell'da)")

In [ ]:
import cdsapi

def download_era5_monthly(year, month, output_dir=ERA5_DIR):
    """
    Bitta oy uchun ERA5 ma'lumotlarini yuklab olish.
    
    Parametrlar:
    - 10m shamol (u va v komponentlari)
    - 2m harorat
    - 2m shudring nuqtasi harorati (namlik uchun)
    - Sirt bosimi
    - Jami yog'in
    - Tuproq namligi (yuqori qatlam)
    """
    output_file = os.path.join(output_dir, f"era5_{year}_{month:02d}.nc")
    
    if os.path.exists(output_file):
        print(f"  ✓ {output_file} allaqachon mavjud, o'tkazildi")
        return output_file
    
    client = cdsapi.Client()
    
    # Kunlik o'rtacha uchun 00, 06, 12, 18 soatlarni olish
    request = {
        'product_type': 'reanalysis',
        'format': 'netcdf',
        'variable': [
            '10m_u_component_of_wind',
            '10m_v_component_of_wind',
            '2m_temperature',
            '2m_dewpoint_temperature',
            'surface_pressure',
            'total_precipitation',
            'volumetric_soil_water_layer_1',
        ],
        'year': str(year),
        'month': f"{month:02d}",
        'day': [f"{d:02d}" for d in range(1, 32)],
        'time': ['00:00', '06:00', '12:00', '18:00'],
        'area': [UZB_BOUNDS['north'], UZB_BOUNDS['west'], 
                 UZB_BOUNDS['south'], UZB_BOUNDS['east']],
    }
    
    print(f"  ⬇️ Yuklanmoqda: ERA5 {year}-{month:02d}...")
    client.retrieve('reanalysis-era5-single-levels', request, output_file)
    print(f"  ✅ Saqlandi: {output_file}")
    
    return output_file

In [ ]:
# ERA5 ni yuklab olish (2018-2020 — Sentinel-5P bilan validatsiya uchun)
# DIQQAT: Bu jarayon uzoq vaqt olishi mumkin (har bir oy ~5-15 daqiqa)
# Birinchi navbatda 2018-2020 ni yuklaymiz, keyin kerak bo'lsa 2011-2017 ham

print("ERA5 yuklab olish (2018-2020):")
print("="*50)

era5_files = []
for year in range(2018, 2021):
    for month in range(1, 13):
        try:
            f = download_era5_monthly(year, month)
            era5_files.append(f)
        except Exception as e:
            print(f"  ❌ {year}-{month:02d}: {e}")

print(f"\n✅ Jami yuklangan: {len(era5_files)} fayl")

## 3. ERA5 ma'lumotlarini qayta ishlash

NetCDF → stansiya nuqtalari uchun kunlik qiymatlar

In [ ]:
import xarray as xr

def process_era5_to_stations(era5_file, stations_df):
    """
    ERA5 NetCDF faylni ochib, har bir stansiya nuqtasi uchun kunlik o'rtacha qiymatlarni olish.
    Eng yaqin grid nuqtasini topadi (nearest neighbor interpolation).
    """
    ds = xr.open_dataset(era5_file)
    
    results = []
    
    for _, station in stations_df.iterrows():
        lat, lon = station['lat'], station['lon']
        
        # Eng yaqin grid nuqtasini topish
        point = ds.sel(latitude=lat, longitude=lon, method='nearest')
        
        # Kunlik o'rtachalar
        daily = point.resample(time='1D').mean()
        
        df_point = daily.to_dataframe().reset_index()
        df_point['station'] = station['station']
        df_point['date'] = df_point['time'].dt.date
        
        results.append(df_point)
    
    ds.close()
    return pd.concat(results, ignore_index=True)


def calculate_wind_speed(df):
    """U va V komponentlaridan shamol tezligini hisoblash."""
    # ERA5 da: u10 = 10m sharqiy shamol, v10 = 10m shimoliy shamol
    if 'u10' in df.columns and 'v10' in df.columns:
        df['era5_wind_speed'] = np.sqrt(df['u10']**2 + df['v10']**2)
        df['era5_wind_dir'] = np.degrees(np.arctan2(-df['u10'], -df['v10'])) % 360
    return df


def calculate_relative_humidity(df):
    """2m harorat va shudring nuqtasidan nisbiy namlikni hisoblash (Magnus formulasi)."""
    if 't2m' in df.columns and 'd2m' in df.columns:
        # Kelvin → Celsius
        t = df['t2m'] - 273.15
        td = df['d2m'] - 273.15
        # Magnus formulasi
        df['era5_rh'] = 100 * np.exp((17.625 * td) / (243.04 + td)) / \
                                np.exp((17.625 * t) / (243.04 + t))
        df['era5_temp'] = t
    return df

In [ ]:
# ERA5 ni barcha stansiya nuqtalariga moslashtirish
# (faqat cho'l + tekislik + tog_oldi stansiyalari)

valid_coords = coords[coords['terrain_type'] != 'tog'].copy()

era5_station_data = []

era5_files_exist = sorted([f for f in os.listdir(ERA5_DIR) if f.endswith('.nc')])

if era5_files_exist:
    print(f"ERA5 fayllarni qayta ishlash: {len(era5_files_exist)} fayl...\n")
    
    for fname in era5_files_exist:
        filepath = os.path.join(ERA5_DIR, fname)
        print(f"  Qayta ishlash: {fname}...", end=" ")
        try:
            df_era5 = process_era5_to_stations(filepath, valid_coords)
            df_era5 = calculate_wind_speed(df_era5)
            df_era5 = calculate_relative_humidity(df_era5)
            era5_station_data.append(df_era5)
            print("✓")
        except Exception as e:
            print(f"❌ {e}")
    
    if era5_station_data:
        df_era5_all = pd.concat(era5_station_data, ignore_index=True)
        df_era5_all.to_csv(f"{RESULTS_DIR}era5_station_daily.csv", index=False)
        print(f"\n✅ ERA5 stansiya ma'lumoti saqlandi: {len(df_era5_all):,} qator")
    else:
        print("\n⚠️ Hech qanday ERA5 fayl qayta ishlanmadi")
else:
    print("⚠️ ERA5 fayllari topilmadi. Yuqoridagi yuklab olish cell'ini run qiling.")
    print("   Yoki qo'lda yuklab oling: https://cds.climate.copernicus.eu")

## 4. Sentinel-5P UVAI (Aerosol Index) bilan validatsiya

### UVAI nima?
- UV Aerosol Index — havoda aerosol (chang, tutun) borligini ko'rsatadi
- UVAI > 1.0 — aerosol aniqlanadi
- UVAI > 2.0 — kuchli aerosol (chang bo'roni ehtimoli yuqori)
- Sentinel-5P (TROPOMI) — 2018-yil apreldan beri

### Google Earth Engine orqali yuklab olish

In [ ]:
import ee

# Google Earth Engine autentifikatsiya
# Birinchi marta: brauzerda ruxsat berish kerak
try:
    ee.Initialize()
    print("✅ Google Earth Engine tayyor!")
except:
    ee.Authenticate()
    ee.Initialize()
    print("✅ Google Earth Engine autentifikatsiya yakunlandi!")

In [ ]:
def get_sentinel5p_uvai(station_lat, station_lon, start_date, end_date, buffer_km=25):
    """
    Bitta stansiya atrofida Sentinel-5P UVAI qiymatlarini olish.
    
    Parameters:
        station_lat, station_lon: stansiya koordinatalari
        start_date, end_date: davr (str: 'YYYY-MM-DD')
        buffer_km: stansiya atrofidagi radius (km)
    
    Returns:
        DataFrame: sana va UVAI o'rtacha qiymati
    """
    # Stansiya nuqtasi atrofida doira (buffer)
    point = ee.Geometry.Point([station_lon, station_lat])
    region = point.buffer(buffer_km * 1000)  # km → m
    
    # Sentinel-5P UVAI kolleksiyasi
    collection = (ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_AER_AI')
                  .filterDate(start_date, end_date)
                  .filterBounds(region)
                  .select('absorbing_aerosol_index'))
    
    # Har bir rasm uchun hudud o'rtachasini hisoblash
    def extract_mean(image):
        mean = image.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=region,
            scale=7000  # S5P resolution ~7km
        )
        return image.set('uvai_mean', mean.get('absorbing_aerosol_index'))
    
    results = collection.map(extract_mean)
    
    # Natijalarni olish
    info = results.aggregate_array('uvai_mean').getInfo()
    dates = results.aggregate_array('system:time_start').getInfo()
    
    df = pd.DataFrame({
        'date': [datetime.fromtimestamp(d/1000).date() for d in dates],
        'uvai': info
    })
    
    return df

In [ ]:
# Tanlangan stansiyalar uchun UVAI yuklab olish (2018-2020)
# Barchasini olish uzoq — avval 10-15 ta muhim stansiyani tanlaymiz

validation_stations = [
    # Cho'l (chang manbai)
    'NUKUS', 'MUJNAK', 'KUNGRAD', 'JASLYK', 'TAHIATAS', 'KOKARAL',
    # Tekislik (shaharlar — ta'sir hududi)
    'Tashkent', 'BUHARA', 'NAVOIY', 'FERGANA', 'Samarkand',
    # Tog' oldi (oraliq)
    'NURATA', 'DJIZAK'
]

print(f"UVAI yuklab olish: {len(validation_stations)} stansiya, 2018-2020...")
print("="*60)

uvai_data = []

for station_name in validation_stations:
    station_info = coords[coords['station'] == station_name]
    if station_info.empty:
        print(f"  ⚠️ {station_name} koordinatalar topilmadi")
        continue
    
    lat = station_info.iloc[0]['lat']
    lon = station_info.iloc[0]['lon']
    
    print(f"  {station_name} ({lat:.2f}, {lon:.2f})...", end=" ")
    
    try:
        df_uvai = get_sentinel5p_uvai(lat, lon, '2018-04-01', '2020-12-31')
        df_uvai['station'] = station_name
        uvai_data.append(df_uvai)
        print(f"✓ {len(df_uvai)} kun")
    except Exception as e:
        print(f"❌ {e}")

if uvai_data:
    df_uvai_all = pd.concat(uvai_data, ignore_index=True)
    df_uvai_all.to_csv(f"{SATELLITE_DIR}sentinel5p_uvai.csv", index=False)
    print(f"\n✅ UVAI ma'lumoti saqlandi: {len(df_uvai_all):,} qator")
else:
    print("\n⚠️ UVAI ma'lumoti yuklanmadi")

## 5. Validatsiya: Label vs UVAI

Bizning chang bo'roni label'larimiz sun'iy yo'ldosh ko'rsatadigan changg mos kelishini tekshirish.

In [ ]:
# UVAI ma'lumotini yuklash (agar avval yuklangan bo'lsa)
uvai_file = f"{SATELLITE_DIR}sentinel5p_uvai.csv"

if os.path.exists(uvai_file):
    df_uvai_all = pd.read_csv(uvai_file, parse_dates=['date'])
    print(f"✅ UVAI yuklandi: {len(df_uvai_all):,} qator")
else:
    print("⚠️ Avval UVAI yuklab olish cell'ini run qiling")

In [ ]:
# Label va UVAI ni birlashtirish
df_val = df_labeled[df_labeled['station'].isin(validation_stations)].copy()
df_val['date'] = pd.to_datetime(df_val['date']).dt.date

df_uvai_all['date'] = pd.to_datetime(df_uvai_all['date']).dt.date

# Birlashtirish
df_validation = df_val.merge(df_uvai_all[['station', 'date', 'uvai']], 
                              on=['station', 'date'], how='inner')

print(f"Validatsiya uchun mos kelgan qatorlar: {len(df_validation):,}")
print(f"\nUVAI statistikasi:")
print(df_validation.groupby('dust_severity')['uvai'].describe().round(2))

In [ ]:
# VALIDATSIYA GRAFIGI: UVAI taqsimoti dust_severity bo'yicha
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Boxplot: UVAI vs dust_severity
order = ['NONE', 'YENGIL', 'ORTACHA', 'KUCHLI']
existing_order = [s for s in order if s in df_validation['dust_severity'].unique()]

sns.boxplot(data=df_validation, x='dust_severity', y='uvai', 
            order=existing_order, ax=axes[0],
            palette={'NONE': '#cccccc', 'YENGIL': '#fdd835', 
                     'ORTACHA': '#ff9800', 'KUCHLI': '#d32f2f'})
axes[0].axhline(y=1.0, color='blue', linestyle='--', alpha=0.5, label='UVAI=1 (aerosol)')
axes[0].axhline(y=2.0, color='red', linestyle='--', alpha=0.5, label='UVAI=2 (kuchli)')
axes[0].set_title('UVAI taqsimoti — chang darajasi bo\'yicha', fontsize=13)
axes[0].set_ylabel('UVAI (Aerosol Index)')
axes[0].legend()

# 2. Scatter: V vs UVAI
scatter_data = df_validation[df_validation['uvai'].notna() & df_validation['V'].notna()]
colors_map = {'NONE': '#cccccc', 'YENGIL': '#fdd835', 'ORTACHA': '#ff9800', 'KUCHLI': '#d32f2f'}
scatter_colors = [colors_map.get(s, '#cccccc') for s in scatter_data['dust_severity']]

axes[1].scatter(scatter_data['V'], scatter_data['uvai'], 
               c=scatter_colors, alpha=0.3, s=10)
axes[1].axhline(y=1.0, color='blue', linestyle='--', alpha=0.5)
axes[1].axvline(x=1.0, color='red', linestyle='--', alpha=0.5)
axes[1].set_xlabel('V — ko\'rinish (km)')
axes[1].set_ylabel('UVAI (Aerosol Index)')
axes[1].set_title('Ko\'rinish vs UVAI (rang = daraja)', fontsize=13)
axes[1].set_xlim(0, 10)

plt.suptitle('VALIDATSIYA: Bizning label vs Sentinel-5P UVAI', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Validatsiya metriklari
print("📊 VALIDATSIYA NATIJALARI:")
print("="*60)

# UVAI > 1 = sun'iy yo'ldosh bo'yicha "chang bor"
df_validation['uvai_dust'] = df_validation['uvai'] > 1.0
df_validation['our_dust'] = df_validation['is_dust']

# Confusion matrix
tp = ((df_validation['our_dust']) & (df_validation['uvai_dust'])).sum()
fp = ((df_validation['our_dust']) & (~df_validation['uvai_dust'])).sum()
fn = ((~df_validation['our_dust']) & (df_validation['uvai_dust'])).sum()
tn = ((~df_validation['our_dust']) & (~df_validation['uvai_dust'])).sum()

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
accuracy = (tp + tn) / len(df_validation)

print(f"\n  Confusion Matrix (UVAI > 1.0 = chang):")
print(f"                  UVAI: Chang    UVAI: Yo'q")
print(f"  Bizda: Chang     {tp:>6}       {fp:>6}")
print(f"  Bizda: Yo'q      {fn:>6}       {tn:>6}")

print(f"\n  Precision: {precision:.3f} (bizda 'chang' deganimizning qanchasi UVAI bilan mos)")
print(f"  Recall:    {recall:.3f} (UVAI ko'rsatgan changning qanchasini biz ushladik)")
print(f"  F1-score:  {f1:.3f}")
print(f"  Accuracy:  {accuracy:.3f}")

print(f"\n  IZOH:")
print(f"  - Precision past = biz 'chang' degan lekin UVAI ko'rsatmagan kunlar ko'p (tuman/smog?)")
print(f"  - Recall past = UVAI chang ko'rsatgan lekin biz ushlamagan kunlar ko'p (V>2 km lekin chang bor?)")
print(f"  - Ideal emas, chunki UVAI kunlik 1 o'lchov, V esa kunlik o'rtacha — vaqt farqi bor")

## 6. Feature Matrix yaratish (ML uchun)

Barcha ma'lumotlarni bitta jadvalga birlashtirish — model uchun tayyor kirish.

In [ ]:
# Feature matrix — barcha ma'lumotlar bitta joyda

# Asosiy: 1-bosqich natijalari
df_features = df_labeled.copy()

# ERA5 qo'shish (agar mavjud)
era5_file = f"{RESULTS_DIR}era5_station_daily.csv"
if os.path.exists(era5_file):
    df_era5 = pd.read_csv(era5_file, parse_dates=['date'])
    df_era5['date'] = df_era5['date'].dt.date
    df_features['date_key'] = pd.to_datetime(df_features['date']).dt.date
    
    era5_cols = ['station', 'date', 'era5_wind_speed', 'era5_wind_dir', 
                 'era5_rh', 'era5_temp', 'sp', 'swvl1', 'tp']
    era5_available = [c for c in era5_cols if c in df_era5.columns]
    
    df_features = df_features.merge(
        df_era5[era5_available], 
        left_on=['station', 'date_key'], 
        right_on=['station', 'date'],
        how='left', suffixes=('', '_era5')
    )
    print(f"✅ ERA5 qo'shildi")

# UVAI qo'shish (agar mavjud)
uvai_file = f"{SATELLITE_DIR}sentinel5p_uvai.csv"
if os.path.exists(uvai_file):
    df_uvai = pd.read_csv(uvai_file, parse_dates=['date'])
    df_uvai['date'] = df_uvai['date'].dt.date
    
    if 'date_key' not in df_features.columns:
        df_features['date_key'] = pd.to_datetime(df_features['date']).dt.date
    
    df_features = df_features.merge(
        df_uvai[['station', 'date', 'uvai']],
        left_on=['station', 'date_key'],
        right_on=['station', 'date'],
        how='left', suffixes=('', '_uvai')
    )
    print(f"✅ UVAI qo'shildi")

# Temporal features (mavsumiy va lag)
df_features['month'] = pd.to_datetime(df_features['date']).dt.month
df_features['day_of_year'] = pd.to_datetime(df_features['date']).dt.dayofyear
df_features['season'] = df_features['month'].map(
    {12:'qish', 1:'qish', 2:'qish', 3:'bahor', 4:'bahor', 5:'bahor',
     6:'yoz', 7:'yoz', 8:'yoz', 9:'kuz', 10:'kuz', 11:'kuz'}
)

# Lag features (har bir stansiya uchun kechagi kunning qiymatlari)
lag_cols = ['V', 'VxG', 'UN', 'Taav']
lag_cols_available = [c for c in lag_cols if c in df_features.columns]

df_features = df_features.sort_values(['station', 'date']).reset_index(drop=True)

for col in lag_cols_available:
    df_features[f'{col}_lag1'] = df_features.groupby('station')[col].shift(1)
    df_features[f'{col}_lag2'] = df_features.groupby('station')[col].shift(2)
    df_features[f'{col}_lag3'] = df_features.groupby('station')[col].shift(3)
    # Rolling mean (3 kunlik o'rtacha)
    df_features[f'{col}_roll3'] = df_features.groupby('station')[col].transform(
        lambda x: x.rolling(3, min_periods=1).mean()
    )

print(f"\n📊 FEATURE MATRIX:")
print(f"   Qatorlar: {len(df_features):,}")
print(f"   Ustunlar: {len(df_features.columns)}")
print(f"   Features: {[c for c in df_features.columns if c not in ['station','date','date_key']]}")

In [ ]:
# Feature matrix saqlash
output_file = f"{RESULTS_DIR}feature_matrix.csv"
df_features.to_csv(output_file, index=False)
print(f"✅ Feature matrix saqlandi: {output_file}")
print(f"   Hajmi: {os.path.getsize(output_file) / 1024 / 1024:.1f} MB")
print(f"\n   Keyingi qadam: 3-bosqich (ML model qurish)")

## 7. Xulosa

### Bu bosqichda qilingan ishlar:
1. ✅ ERA5 reanaliz ma'lumotlari (shamol, harorat, namlik, yog'in, tuproq namligi)
2. ✅ Sentinel-5P UVAI bilan validatsiya
3. ✅ Feature matrix (ML uchun tayyor)

### Keyingi qadam (3-bosqich):
- XGBoost/LightGBM model qurish
- Train: 2011-2018, Test: 2019-2020
- 3 kunlik prognoz (target: keyingi 1/2/3 kunda chang bo'ladimi?)
- Feature importance tahlili